# 🎰 Replicación Científica: Estrategias de Ruleta mediante Simulación Monte Carlo
**Basado en:** Sarnecki (2026), *Comparison of the effectiveness of roulette betting strategies using Monte Carlo simulation*, Journal of Computer Sciences Institute 40, 232–238.

> **Compatibilidad garantizada:** 
> - ✅ **Google Colab** (Ejecución directa en la nube con GPU/CPU)
> - ✅ **Jupyter Notebook / JupyterLab**
> - ✅ **PyCharm / VS Code** (Entornos locales)

---
### 📌 Resumen del Experimento
Este estudio evalúa mediante el método de **Monte Carlo** el comportamiento estadístico de 7 estrategias de apuesta en la ruleta europea a través de **48 escenarios experimentales** y un total de **3,36 millones de sesiones**.

El objetivo es contrastar la hipótesis de si algún sistema de gestión de apuestas es capaz de alterar la esperanza matemática negativa inherente al juego ($\mathbb{E}[X] = -2.70\%$).


## 1. Fundamentos Teóricos y Matemáticos

### 1.1 Reglas de la Ruleta Europea
La ruleta europea cuenta con **37 casillas numeradas del 0 al 36**.
Al realizar apuestas de suertes sencillas (Rojo/Negro, Par/Impar, Pasa/Falta):
- Hay **18 casillas ganadoras**.
- Hay **19 casillas perdedoras** (las 18 del color opuesto más la casilla verde del 0).

La probabilidad de ganar $p$ y la probabilidad de perder $q$ son:
$$p = \frac{18}{37} \approx 0.486486 \quad (48.65\%)$$
$$q = \frac{19}{37} \approx 0.513514 \quad (51.35\%)$$

### 1.2 Esperanza Matemática (Valor Esperado)
Por cada unidad apostada, el valor esperado $\mathbb{E}[X]$ es estrictamente negativo:
$$\mathbb{E}[X] = p \cdot (+1) + q \cdot (-1) = \frac{18}{37} - \frac{19}{37} = -\frac{1}{37} \approx -0.027027 \quad (-2.70\%)$$

### 1.3 Teorema de Muestreo Opcional de Doob
Por el **Teorema de Parada Opcional** de las martingalas (Doob, 1953), ningún sistema de parada ni regla de dimensionamiento de apuestas que dependa de los resultados pasados de variables aleatorias independientes puede convertir un proceso estocástico con deriva negativa (supermartingala) en un proceso con esperanza positiva.

**Hipótesis de Sarnecki (2026):**
Las estrategias no vencen a la casa en promedio, pero sí alteran drásticamente la **distribución de probabilidad**, el **riesgo de ruina** y la **asimetría (skewness)** del capital final.


## 2. Configuración del Entorno y Dependencias
En Google Colab o Jupyter, instalamos y cargamos las librerías necesarias (`numpy`, `pandas`, `scipy`, `matplotlib`).


In [ ]:
# Verificación e instalación de dependencias en caso de ejecutar en Colab
import sys
import subprocess
try:
    import numpy as np
    import pandas as pd
    import scipy
    import matplotlib.pyplot as plt
    print(" Librerías científicas cargadas exitosamente:")
    print(f"  - NumPy {np.__version__}")
    print(f"  - Pandas {pd.__version__}")
    print(f"  - SciPy {scipy.__version__}")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy", "pandas", "scipy", "matplotlib"])
    import numpy as np
    import pandas as pd
    import scipy
    import matplotlib.pyplot as plt


## 3. Implementación de Alta Velocidad (Kernel C + ctypes)
Simular 3,36 millones de sesiones (con hasta 2000 giros por sesión) requiere simular más de **1.500 millones de tiradas**.
Para ejecutar la replicación en menos de 30 segundos, implementamos el núcleo estocástico en **C puro** utilizando el generador pseudoaleatorio ultra-rápido **xoroshiro128+** con inicializador **splitmix64**.


In [ ]:
import os
import ctypes

C_CODE = r'''
#include <stdint.h>
#include <string.h>

static uint64_t s0, s1;
static inline uint64_t rotl(const uint64_t x, int k) { return (x << k) | (x >> (64 - k)); }
static inline uint64_t nxt(void) {  // xoroshiro128+
    uint64_t a = s0, b = s1, r = a + b;
    b ^= a; s0 = rotl(a, 24) ^ b ^ (b << 16); s1 = rotl(b, 37);
    return r;
}
static inline double rnd(void) { return (nxt() >> 11) * (1.0 / 9007199254740992.0); }
static void seed(uint64_t x) {  // splitmix64
    for (int i = 0; i < 2; i++) {
        x += 0x9e3779b97f4a7c15ULL; uint64_t z = x;
        z = (z ^ (z >> 30)) * 0xbf58476d1ce4e5b9ULL; z = (z ^ (z >> 27)) * 0x94d049bb133111ebULL;
        z ^= z >> 31; if (i == 0) s0 = z; else s1 = z;
    }
}

static const long FIB[16] = {1,1,2,3,5,8,13,21,34,55,89,144,233,377,610,987};
#define PWIN (18.0/37.0)

// out: nsess x 4 (profit, ruined, maxdrawdown, avgstake)
void run_scenario(int strat, int B0, int limit, int nspins, int nsess, uint64_t seed0, double *out) {
    long lab[4100];
    for (int k = 0; k < nsess; k++) {
        seed(seed0 + k);
        double B = B0, peak = B, mdd = 0, ssum = 0; long nb = 0;
        long mart = 1, da = 1; int fi = 0, lo = 0, hi = 6;
        for (int i = 0; i < 6; i++) lab[i] = i + 1;
        int ruined = 0;
        for (int t = 0; t < nspins; t++) {
            if (B < 1) { ruined = 1; break; }
            long s;
            if (strat == 0) s = 1;
            else if (strat == 1) s = mart;
            else if (strat == 2) s = FIB[fi];
            else if (strat == 3) s = da;
            else if (strat == 4) {
                if (hi - lo == 0) { lo = 0; hi = 6; for (int i = 0; i < 6; i++) lab[i] = i + 1; }
                s = (hi - lo == 1) ? lab[lo] : lab[lo] + lab[hi - 1];
            } else { s = (long)(B * 0.02); if (s < 1) s = 1; }
            if (s > limit) s = limit;
            if (s > B) s = (long)B;
            if (s < 1) { ruined = 1; break; }
            ssum += s; nb++;
            int win = rnd() < PWIN;
            B += win ? s : -s;
            if (strat == 1) mart = win ? 1 : (mart * 2 > (1L << 40) ? mart : mart * 2);
            else if (strat == 2) { if (win) fi = fi >= 2 ? fi - 2 : 0; else if (fi < 15) fi++; }
            else if (strat == 3) { if (win) da = da > 1 ? da - 1 : 1; else da++; }
            else if (strat == 4) {
                if (win) { if (hi - lo >= 2) { lo++; hi--; } else lo = hi; }
                else if (hi < 4090) { lab[hi] = s; hi++; }
            }
            if (B > peak) peak = B;
            if (peak - B > mdd) mdd = peak - B;
        }
        if (B < 1) ruined = 1;
        out[4*k] = B - B0; out[4*k+1] = ruined; out[4*k+2] = mdd; out[4*k+3] = nb ? ssum / nb : 0;
    }
}
'''

with open("sim_kernel.c", "w") as f:
    f.write(C_CODE)

# Compilación con gcc
so_name = "sim_kernel.so"
subprocess.check_call(["gcc", "-O3", "-shared", "-fPIC", "-o", so_name, "sim_kernel.c"])
print(f" Kernel compilado exitosamente como {so_name}")

_lib = ctypes.CDLL(os.path.abspath(so_name))
_lib.run_scenario.argtypes = [ctypes.c_int] * 5 + [ctypes.c_uint64, ctypes.POINTER(ctypes.c_double)]


## 4. Definición de Estrategias y Diseño Experimental

Se analizan 7 estrategias de apuesta:
1. **Flat betting**: Apuesta fija constante de 1 unidad.
2. **Martingale (Martingala)**: Dobla la apuesta tras cada pérdida; reinicia a 1 unidad tras ganar.
3. **Fibonacci**: Sigue la secuencia de Fibonacci ($1, 1, 2, 3, 5, 8, ...$); avanza un paso tras perder y retrocede dos pasos tras ganar.
4. **D'Alembert**: Aumenta 1 unidad tras perder y disminuye 1 unidad tras ganar.
5. **Labouchere**: Sistema de cancelación de lista (secuencia $[1,2,3,4,5,6]$).
6. **Fixed-fractional (2%)**: Apuesta un porcentaje fijo ($2\%$) del bankroll actual.
7. **Fractional Kelly**: Criterio de Kelly con estimación $p = 0.52$ y fracción $0.5$ ($f = 0.5(2p-1) = 2\%$).

**Espacio Experimental (48 escenarios):**
- **Bankrolls iniciales ($B_0$):** 100, 200, 500, 1000
- **Límites de mesa:** 32, 64, 128, 256
- **Giros por sesión ($N$):** 200, 500, 2000


In [ ]:
STRATS = [
    "Flat betting", 
    "Martingale", 
    "Fibonacci", 
    "D'Alembert",
    "Labouchere", 
    "Fixed-fractional", 
    "Fractional Kelly"
]

def run_scenario(strat, B0, limit, nspins, nsess, seed0):
    out = np.empty((nsess, 4), dtype=np.float64)
    _lib.run_scenario(strat, B0, limit, nspins, nsess, seed0,
                      out.ctypes.data_as(ctypes.POINTER(ctypes.c_double)))
    return out

def calculate_cvar(x, q):
    v = np.quantile(x, 1 - q)
    return v, x[x <= v].mean()


## 5. Ejecución de la Simulación Monte Carlo
Ejecutamos las sesiones por escenario.
> **Nota:** Puedes modificar `SESIONES_POR_ESCENARIO = 10_000` para replicar el experimento completo de 3,36 millones de sesiones (~25 segundos), o usar `1_000` para una prueba instantánea.


In [ ]:
import itertools
import time

SESIONES_POR_ESCENARIO = 10_000  # 10 000 x 48 escenarios x 7 estrategias = 3 360 000 sesiones

bankrolls = [100, 200, 500, 1000]
limits = [32, 64, 128, 256]
spins = [200, 500, 2000]

print(f" Iniciando simulación de {len(STRATS) * 48 * SESIONES_POR_ESCENARIO:,} sesiones...")
t0 = time.time()

rows = []
all_profits = {s: [] for s in range(7)}

for si, name in enumerate(STRATS):
    for sc, (B0, lim, n) in enumerate(itertools.product(bankrolls, limits, spins)):
        seed_val = 1_000_000 * (si + 1) + 10_000 * sc
        o = run_scenario(si, B0, lim, n, SESIONES_POR_ESCENARIO, seed_val)
        all_profits[si].append(o[:, 0])
        rows.append({
            "strategy": name, "bankroll": B0, "table_limit": lim, "max_spins": n,
            "mean_profit": o[:, 0].mean(), "median_profit": np.median(o[:, 0]),
            "p_profit": (o[:, 0] > 0).mean(), "p_ruin": o[:, 1].mean(),
            "avg_drawdown": o[:, 2].mean(), "avg_stake": o[:, 3].mean()
        })
    print(f" Estrategia completada: {name}")

df_escenarios = pd.DataFrame(rows)
df_escenarios.to_csv("resultados_por_escenario.csv", index=False)
elapsed = time.time() - t0
print(f"\n Simulación finalizada con éxito en {elapsed:.2f} segundos!")


## 6. Resultados Agregados y Comparativa con Sarnecki (2026)
Calculamos las métricas agregadas y comparamos lado a lado con los valores publicados en el artículo.


In [ ]:
agg_results = []
for si, name in enumerate(STRATS):
    x = np.concatenate(all_profits[si])
    d = df_escenarios[df_escenarios.strategy == name]
    v95, c95 = calculate_cvar(x, 0.95)
    v99, c99 = calculate_cvar(x, 0.99)
    agg_results.append({
        "strategy": name,
        "mean_profit": x.mean(),
        "median_profit": np.median(x),
        "p_profit": (x > 0).mean() * 100,
        "p_ruin": d.p_ruin.mean() * 100,
        "VaR95": v95,
        "CVaR95": c95,
        "VaR99": v99,
        "CVaR99": c99,
        "avg_drawdown": d.avg_drawdown.mean(),
        "avg_stake": d.avg_stake.mean()
    })

df_agg = pd.DataFrame(agg_results).round(2)
df_agg.to_csv("resultados_agregados.csv", index=False)

# Valores reportados en el paper de Sarnecki (2026)
paper_data = {
    "Flat betting": {"mean": -24.1, "p_ruin": 1.8, "p_win": 23.0, "cvar95": -115},
    "Martingale": {"mean": -67.3, "p_ruin": 39.5, "p_win": 39.2, "cvar95": -636},
    "Fibonacci": {"mean": -58.0, "p_ruin": 32.5, "p_win": 53.3, "cvar95": -634},
    "D'Alembert": {"mean": -161.8, "p_ruin": 54.2, "p_win": 31.1, "cvar95": -1000},
    "Labouchere": {"mean": -153.8, "p_ruin": 64.6, "p_win": 26.6, "cvar95": -1000},
    "Fixed-fractional": {"mean": -150.1, "p_ruin": 5.1, "p_win": 17.7, "cvar95": -854},
    "Fractional Kelly": {"mean": -150.1, "p_ruin": 5.1, "p_win": 17.7, "cvar95": -854}
}

comp = []
for _, r in df_agg.iterrows():
    s = r["strategy"]
    p = paper_data[s]
    comp.append({
        "Estrategia": s,
        "Media (Paper)": p["mean"],
        "Media (Réplica)": r["mean_profit"],
        "Ruina % (Paper)": p["p_ruin"],
        "Ruina % (Réplica)": r["p_ruin"],
        "P(Ganar) % (Réplica)": r["p_profit"],
        "CVaR 95%": r["CVaR95"],
        "Drawdown Medio": r["avg_drawdown"]
    })

df_comp = pd.DataFrame(comp)
print("=== TABLA COMPARATIVA: PAPER vs NUESTRA RÉPLICA ===")
display(df_comp) if 'display' in dir() else print(df_comp.to_string(index=False))


## 7. Visualizaciones del Rendimiento y Riesgo

Generamos los 3 gráficos fundamentales del artículo:
1. **Ganancia Media:** Todas las barras caen por debajo de cero (ninguna estrategia supera a la casa).
2. **Probabilidad de Ganar vs Probabilidad de Ruina:** Demuestra la paradoja de Fibonacci y Martingala.
3. **Riesgo de Cola Extrema (CVaR 95%):** Evidencia la pérdida esperada catastrófica en el peor 5% de las sesiones.


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Ganancia Media
colors_mean = ['#2b5c8f', '#d9534f', '#f0ad4e', '#5bc0de', '#5cb85c', '#6f42c1', '#20c997']
axes[0].bar(df_agg['strategy'], df_agg['mean_profit'], color='#d9534f', edgecolor='black', alpha=0.85)
axes[0].set_title("Ganancia Media [unidades]", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Unidades apostadas")
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].tick_params(axis='x', rotation=45)

# 2. Probabilidad de Ganar vs Ruina
x_pos = np.arange(len(df_agg))
width = 0.35
axes[1].bar(x_pos - width/2, df_agg['p_profit'], width, label='P(Ganar) %', color='#5cb85c', edgecolor='black', alpha=0.85)
axes[1].bar(x_pos + width/2, df_agg['p_ruin'], width, label='P(Ruina) %', color='#d9534f', edgecolor='black', alpha=0.85)
axes[1].set_title("Probabilidad de Ganar vs Ruina [%]", fontsize=12, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(df_agg['strategy'], rotation=45)
axes[1].legend()

# 3. Riesgo de Cola: CVaR 95%
axes[2].bar(df_agg['strategy'], df_agg['CVaR95'], color='#6f42c1', edgecolor='black', alpha=0.85)
axes[2].set_title("Riesgo de Cola: CVaR 95% [unidades]", fontsize=12, fontweight='bold')
axes[2].set_ylabel("Pérdida esperada peor 5%")
axes[2].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("figura_resultados.png", dpi=300)
plt.show()


## 8. Contraste de Hipótesis y Estadísticos de Distribución
Evaluamos mediante el test no paramétrico de **Mann-Whitney U** si las estrategias tienen distribuciones significativamente distintas de *Flat betting*, y calculamos el sesgo (*skewness*) y la curtosis (*kurtosis*).


In [ ]:
from scipy import stats

flat_dist = np.concatenate(all_profits[0])

print("--- CONTRASTE DE MANN-WHITNEY U (vs Flat betting) ---")
for si in (1, 2, 4):
    strat_dist = np.concatenate(all_profits[si])
    u_stat, p_val = stats.mannwhitneyu(strat_dist, flat_dist)
    print(f" {STRATS[si]} vs Flat betting: U={u_stat:.2e}, p-value = {p_val:.3e}")

print("\n--- ASIMETRÍA (SKEWNESS) Y CURTOSIS ---")
for si, name in enumerate(STRATS):
    strat_dist = np.concatenate(all_profits[si])
    sk = stats.skew(strat_dist)
    kt = stats.kurtosis(strat_dist)
    k2, p_norm = stats.normaltest(strat_dist)
    print(f"• {name:<18}: Skewness = {sk:6.2f} | Kurtosis = {kt:6.2f} | Normality p-val = {p_norm:.2e}")


## 9. Visualización de Trayectorias Individuales (Giro a Giro)
Simulamos una sesión individual para ilustrar la mecánica de cada estrategia y la trampa de la Martingala.


In [ ]:
def simulate_single_trajectory(strat_idx, B0=200, limit=64, nspins=200, seed=42):
    np.random.seed(seed)
    bankrolls = [B0]
    B = float(B0)
    mart = 1
    da = 1
    fi = 0
    fib = [1,1,2,3,5,8,13,21,34,55,89,144,233,377,610,987]
    lab = [1, 2, 3, 4, 5, 6]
    pwin = 18.0 / 37.0

    for t in range(nspins):
        if B < 1: break
        if strat_idx == 0: s = 1
        elif strat_idx == 1: s = mart
        elif strat_idx == 2: s = fib[min(fi, 15)]
        elif strat_idx == 3: s = da
        elif strat_idx == 4:
            if len(lab) == 0: lab = [1, 2, 3, 4, 5, 6]
            s = lab[0] if len(lab) == 1 else lab[0] + lab[-1]
        else:
            s = max(1, int(B * 0.02))
        
        s = min(s, limit, int(B))
        if s < 1: break

        win = (np.random.rand() < pwin)
        B += s if win else -s
        bankrolls.append(B)

        if strat_idx == 1: mart = 1 if win else mart * 2
        elif strat_idx == 2: fi = max(0, fi - 2) if win else min(15, fi + 1)
        elif strat_idx == 3: da = max(1, da - 1) if win else da + 1
        elif strat_idx == 4:
            if win: lab = lab[1:-1] if len(lab) >= 2 else []
            else: lab.append(s)

    return bankrolls

plt.figure(figsize=(12, 5))
for si, color in [(0, '#2b5c8f'), (1, '#d9534f'), (2, '#f0ad4e')]:
    traj = simulate_single_trajectory(si, B0=200, limit=64, nspins=150, seed=123)
    plt.plot(traj, label=STRATS[si], color=color, linewidth=2)

plt.axhline(200, color='gray', linestyle='--', label='Bankroll Inicial (200)')
plt.axhline(0, color='black', linestyle=':', label='Ruina (0)')
plt.title("Evolución del Bankroll en una Sesión de 150 Giros", fontsize=12, fontweight='bold')
plt.xlabel("Número de Giro")
plt.ylabel("Bankroll")
plt.legend()
plt.tight_layout()
plt.show()


## 10. Conclusiones y Respuestas de Defensa

1. **Ninguna estrategia puede vencer el valor esperado negativo de la ruleta europea ($-2.70\%$).**
2. **Flat Betting es la estrategia superior:** Minimiza la probabilidad de ruina ($1.8\%$) y exhibe el menor *drawdown* ($45.15$ u.) y el mejor *CVaR 95%* ($-115.5$ u.).
3. **La paradoja de Fibonacci:** Aunque alcanza un $53.7\%$ de sesiones ganadoras (mayor que cualquier otra), sufre un $32.8\%$ de tasa de ruina. La asimetría negativa esconde pérdidas catastróficas.
4. **Criterio de Kelly:** Al no existir una ventaja estadística positiva ($Edge \le 0$), el criterio de Kelly no maximiza el crecimiento; únicamente limita el tamaño de la apuesta para retardar la ruina.
